# MSML606 HW-3

### AI tools (ChatGPT, Claude) were used for general understanding and clarification of concepts (trees, stacks).

## Part I: Hash table implementation

## Problem 1. Implement a hash table.

In [1]:
import time
import random
import matplotlib.pyplot as plt

class HashMap:
    def __init__(self, size=101):
        self.size = size
        self.table = []
        self.count = 0
        self.hash_method = "division"
        for _ in range(size):
            self.table.append([])   # each is a list

    def _key_to_int(self, key):
        # convert key into a number
        if type(key) == int:
            return abs(key)
        elif type(key) == str:
            total = 0
            for ch in key:
                total = total + ord(ch)
            return total
        else:
            # edge case
            text = str(key)
            total = 0
            for ch in text:
                total = total + ord(ch)
            return total

    # retrieve value
    def search(self, key):
        index = self._hash(key, self.hash_method)
        chain = self.table[index]
        for pair in chain:
            if pair[0] == key:
                return pair[1]
        return None

    # insert into the hash table
    def insert(self, key, value):
        index = self._hash(key, self.hash_method)
        chain = self.table[index]
        # update if key already exists
        for i in range(len(chain)):
            if chain[i][0] == key:
                chain[i] = (key, value)
                return
        chain.append((key, value))
        self.count += 1

    # remove the key value pair from table
    def delete(self, key):
        index = self._hash(key, self.hash_method)
        chain = self.table[index]

        for i in range(len(chain)):
            if chain[i][0] == key:
                del chain[i]
                self.count -= 1
                return True

        return False

    def dynamicResizing(self):
        pass

    # hashing methods
    def _hash(self, key, method="division"):
        num = self._key_to_int(key)

        # division
        if method == "division":
            return num % self.size

        # multiplication
        elif method == "multiplication":
            a = 0.618
            frac = (num * a) % 1
            return int(self.size * frac)

        # default case
        return num % self.size

    def load_factor(self):
        return self.count / self.size

    def average_chain_length(self):
        total = 0
        for chain in self.table:
            total += len(chain)
        return total / self.size

    def max_chain_length(self):
        longest = 0
        for chain in self.table:
            if len(chain) > longest:
                longest = len(chain)
        return longest



## Problem 2. Performance Analysis

In [4]:
def generate_keys(distribution, n):
    keys = []

    if distribution == "uniform":
        for _ in range(n):
            keys.append(random.randint(1, 1000000))

    elif distribution == "skewed":
        for _ in range(n):
            if random.random() < 0.8:
                keys.append(random.randint(1, 100))
            else:
                keys.append(random.randint(101, 1000000))

    elif distribution == "sequential":
        for i in range(n):
            keys.append(i + 1)

    return keys


def measure_search_time(hashmap, keys):
    start = time.perf_counter()

    for key in keys:
        hashmap.search(key)

    end = time.perf_counter()

    if len(keys) == 0:
        return 0

    return (end - start) / len(keys)


def run_experiments():

    table_sizes = [101, 503, 1009]
    load_factors = [0.25, 0.50, 0.75, 1.0, 1.25]
    methods = ["division", "multiplication"]
    # Experiment 1
    # Load factor vs search time
    for method in methods:
        for size in table_sizes:
            good_times = []
            bad_times = []

            for lf in load_factors:
                h = HashMap(size)
                h.hash_method = method

                n = int(size * lf)
                keys = generate_keys("uniform", n)

                for key in keys:
                    h.insert(key, key * 10)

                # successful search
                if len(keys) > 100:
                    good_keys = random.sample(keys, 100)
                else:
                    good_keys = keys[:]

                # unsuccessful search
                bad_keys = []
                while len(bad_keys) < len(good_keys):
                    x = random.randint(2000000, 3000000)
                    bad_keys.append(x)

                t_good = measure_search_time(h, good_keys)
                t_bad = measure_search_time(h, bad_keys)

                good_times.append(t_good)
                bad_times.append(t_bad)

                print("Method:", method,
                      "| Size:", size,
                      "| Load Factor:", round(lf, 2),
                      "| Success Time:", t_good,
                      "| Fail Time:", t_bad)

            plt.figure(figsize=(7, 5))
            plt.plot(load_factors, good_times, marker='o', label='successful search')
            plt.plot(load_factors, bad_times, marker='s', label='unsuccessful search')
            plt.xlabel("Load Factor")
            plt.ylabel("Average Search Time (seconds)")
            plt.title("Load Factor vs Search Time (" + method + ", size=" + str(size) + ")")
            plt.legend()
            plt.grid(True)
            plt.show()
